# AUTOMATIZZAZIONE MINIMIZZAZIONE NLL

---

Leggo file, predispongo parametri per il noising poissoniano

In [ ]:
import os
import time
from datetime import datetime
from itertools import product
import h5py
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from ml_odmr.FitMonteCarlo import FitMonteCarlo
from ml_odmr.PoissonNoise import PoissonNoise

# =========================================================================
# 1. PARAMETRI DELLA GRIGLIA DA ESEGUIRE 
# =========================================================================
list_contrast = [0.8, 0.5, 0.2, 0.1, 0.08, 0.05, 0.02, 0.01]
list_Nph = [3e5, 6e5, 12e5, 24e5, 48e5, 60e5, 120e5, 240e5, 480e5]
list_mc_trials = [1, 5, 10, 20, 40]

mat_file_path = "../data/10k/odmr_easyspin_raw_10000.mat"
# Percorso nella root del workspace (allo stesso livello di 'data/')
base_output_dir = os.path.abspath("../benchmarks/10k")
os.makedirs(base_output_dir, exist_ok=True)

random_seed = 42

# Caricamento dati dal file .mat
with h5py.File(mat_file_path, "r") as f:
    spectra_data = np.array(f["spectra_data"]).T
    mw_freqs = np.array(f["mw_freqs"]).squeeze()
    labels_c = np.array(f["labels_c"]).squeeze()
    labels_s = np.array(f["labels_s"]).squeeze()

f_ghz = np.asarray(mw_freqs, dtype=np.float64)
N_spectra_to_run = len(spectra_data)

noise_gen = PoissonNoise(seed=random_seed)


## PROVA SINGOLO FIT

## PROVA FIT AUTOMATIZZATO

In [3]:
import gc
import multiprocessing as mp
import os
import time
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd

from iminuit import Minuit
from ml_odmr.FitLikelihood import worker_chunk_nll, create_nll

# =========================================================================
# 1.1 CONFIGURAZIONE PERCORSI
# =========================================================================
list_contrast = [0.8, 0.5, 0.2, 0.1, 0.08, 0.05, 0.02, 0.01]
list_Nph = [3e5, 6e5, 12e5, 24e5, 48e5, 60e5, 120e5, 240e5, 480e5]
list_mc_trials = [1, 5, 10, 20, 40]

base_nll_dir = os.path.abspath("../benchmarks/10k/likelihood")
os.makedirs(base_nll_dir, exist_ok=True)

# Asse frequenze normalizzato (calcolato una sola volta)
f_min, f_max = float(f_ghz.min()), float(f_ghz.max())
delta_f = f_max - f_min
f_norm_global = np.ascontiguousarray((f_ghz - f_min) / delta_f, dtype=np.float64)

# =========================================================================
# 2. ESECUZIONE A CHUNK CON MULTIPROCESSING POOL (RAM SOTTO CONTROLLO)
# =========================================================================
N_WORKERS = 4
N_CHUNKS = 16  # 16 blocchi da 625 spettri ciascuno

index_chunks = np.array_split(np.arange(N_spectra_to_run), N_CHUNKS)
all_combinations = list(product(list_mc_trials, list_contrast, list_Nph))
total_combos = len(all_combinations)

print(f"Totale combinazioni NLL: {total_combos}")
print(f"Spettri per combinazione: {N_spectra_to_run}")
print(f"Cartella di output: '{base_nll_dir}'\n")

ctx = mp.get_context("spawn")
t_batch_start = time.time()

for combo_idx, (m_val, c_val, n_ph) in enumerate(all_combinations, start=1):
    sub_dir = os.path.join(base_nll_dir, f"contrast={c_val}", f"mc_trials={m_val}")
    os.makedirs(sub_dir, exist_ok=True)
    filepath = os.path.join(sub_dir, f"fit_nll_Nph{int(n_ph)}.csv")

    if os.path.exists(filepath):
        print(f"[{combo_idx}/{total_combos}] SALTO (esistente): C={c_val}, M={m_val}, N_ph={n_ph:.1e}")
        continue

    print(f"[{combo_idx}/{total_combos}] Avvio NLL: C={c_val:<4} | M={m_val:<2} | N_ph={n_ph:.1e} ...", end=" ", flush=True)
    t_combo_start = time.time()

    tasks = [
        (
            ch, spectra_data[ch], labels_c[ch], labels_s[ch],
            f_norm_global, f_min, delta_f, c_val, n_ph, m_val, random_seed,
        )
        for ch in index_chunks
    ]

    # Pool reale: alla fine del blocco 'with' i 4 processi vengono chiusi e la RAM liberata
    with ctx.Pool(processes=N_WORKERS) as pool:
        chunks_out = pool.map(worker_chunk_nll, tasks)

    results_dicts = [item for sublist in chunks_out for item in sublist]
    combo_wall_time = time.time() - t_combo_start

    df_res = pd.DataFrame(results_dicts)
    n_conv = int(df_res["converged"].sum())
    succ_rate = (n_conv / N_spectra_to_run) * 100.0
    mae_c = df_res["err_c_norm"].mean()
    mae_s = df_res["err_s_norm"].mean()
    mean_t = df_res["fit_time_sec"].mean()

    header_metadata = (
        f"# ODMR Poisson NLL (iminuit) Monte Carlo Fit Benchmark\n"
        f"# Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"# Source File: {os.path.basename(mat_file_path)}\n"
        f"# N_spectra: {N_spectra_to_run}\n"
        f"# N_photons: {n_ph:.2e}\n"
        f"# Contrast: {c_val}\n"
        f"# MC_Trials_M: {m_val}\n"
        f"# Base_Seed: {random_seed}\n"
        f"# Converged_Spectra: {n_conv}/{N_spectra_to_run} ({succ_rate:.2f}%)\n"
        f"# Mean_MAE_c_norm: {mae_c:.6f}\n"
        f"# Mean_MAE_s_norm: {mae_s:.6f}\n"
        f"# Mean_Fit_Time_Per_Spectrum_s: {mean_t:.4f}\n"
        f"# Total_Wall_Time_s: {combo_wall_time:.2f}\n"
    )

    with open(filepath, mode="w", encoding="utf-8") as f:
        f.write(header_metadata)
        df_res.to_csv(f, index=False, float_format="%.6f")

    elapsed_total_min = (time.time() - t_batch_start) / 60.0
    print(f"Fatto in {combo_wall_time:.1f}s | Conv: {succ_rate:.1f}% | MAE_c: {mae_c:.4f} | MAE_s: {mae_s:.4f} | (Tot: {elapsed_total_min:.1f} min)")

    del tasks, chunks_out, results_dicts, df_res
    gc.collect()

print(f"\nBenchmark NLL completato in {(time.time() - t_batch_start)/60:.1f} minuti!")

Totale combinazioni NLL: 360
Spettri per combinazione: 10000
Cartella di output: '/Users/zosojack/ml-odmr/benchmarks/10k/likelihood'

[1/360] SALTO (esistente): C=0.8, M=1, N_ph=3.0e+05
[2/360] SALTO (esistente): C=0.8, M=1, N_ph=6.0e+05
[3/360] SALTO (esistente): C=0.8, M=1, N_ph=1.2e+06
[4/360] SALTO (esistente): C=0.8, M=1, N_ph=2.4e+06
[5/360] SALTO (esistente): C=0.8, M=1, N_ph=4.8e+06
[6/360] SALTO (esistente): C=0.8, M=1, N_ph=6.0e+06
[7/360] Avvio NLL: C=0.8  | M=1  | N_ph=1.2e+07 ... 

Fatto in 6.4s | Conv: 100.0% | MAE_c: 0.0231 | MAE_s: 0.0303 | (Tot: 0.1 min)
[8/360] Avvio NLL: C=0.8  | M=1  | N_ph=2.4e+07 ... Fatto in 6.0s | Conv: 100.0% | MAE_c: 0.0218 | MAE_s: 0.0285 | (Tot: 0.2 min)
[9/360] Avvio NLL: C=0.8  | M=1  | N_ph=4.8e+07 ... Fatto in 6.2s | Conv: 100.0% | MAE_c: 0.0197 | MAE_s: 0.0254 | (Tot: 0.3 min)
[10/360] SALTO (esistente): C=0.5, M=1, N_ph=3.0e+05
[11/360] SALTO (esistente): C=0.5, M=1, N_ph=6.0e+05
[12/360] SALTO (esistente): C=0.5, M=1, N_ph=1.2e+06
[13/360] SALTO (esistente): C=0.5, M=1, N_ph=2.4e+06
[14/360] SALTO (esistente): C=0.5, M=1, N_ph=4.8e+06
[15/360] SALTO (esistente): C=0.5, M=1, N_ph=6.0e+06
[16/360] Avvio NLL: C=0.5  | M=1  | N_ph=1.2e+07 ... Fatto in 6.4s | Conv: 100.0% | MAE_c: 0.0198 | MAE_s: 0.0273 | (Tot: 0.4 min)
[17/360] Avvio NLL: C=0.5  | M=1  | N_ph=2.4e+07 ... Fatto in 6.4s | Conv: 100.0% | MAE_c: 0.0176 | MAE_s: 0.0246 | (Tot: 0.5 min)
[18/360] Avvio NLL: C=0.5  | M=1  | N_ph=4.8e+07 ... Fatto in 9.4s | Conv: 100.0% 

Process SpawnPoolWorker-162:
Process SpawnPoolWorker-161:
Process SpawnPoolWorker-163:
Process SpawnPoolWorker-164:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Users/zosojack/coding-with-qiskit/miniconda3/envs/ml-seda-env/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/zosojack/coding-with-qiskit/miniconda3/envs/ml-seda-env/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/zosojack/coding-with-qiskit/miniconda3/envs/ml-seda-env/lib/python3.12/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
                    ^^^^^^^^^^^^^^^^^^^
  File "/Users/zosojack/coding-with-qiskit/miniconda3/envs/ml-seda-env/lib/python3.12/multiprocessing/pool.py", line 48, in mapstar
    return list(map(*args))
           ^^^^^^^^^^^^^^^^
  File "/Us

KeyboardInterrupt: 

### primo giro
````
Totale combinazioni nel subset NLL: 240
Spettri per combinazione: 10000
Cartella di output: '/Users/zosojack/ml-odmr/benchmarks/10k/likelihood'

[1/240] Avvio NLL: C=0.8  | M=1  | N_ph=3.0e+05 ... Fatto in 5.9s | Conv: 99.9% | MAE_c: 0.0303 | MAE_s: 0.0387 | (Tot: 0.1 min)
[2/240] Avvio NLL: C=0.8  | M=1  | N_ph=6.0e+05 ... Fatto in 3.8s | Conv: 99.9% | MAE_c: 0.0280 | MAE_s: 0.0362 | (Tot: 0.2 min)
[3/240] Avvio NLL: C=0.8  | M=1  | N_ph=1.2e+06 ... Fatto in 3.7s | Conv: 99.8% | MAE_c: 0.0268 | MAE_s: 0.0346 | (Tot: 0.2 min)
[4/240] Avvio NLL: C=0.8  | M=1  | N_ph=2.4e+06 ... Fatto in 4.2s | Conv: 99.8% | MAE_c: 0.0249 | MAE_s: 0.0320 | (Tot: 0.3 min)
[5/240] Avvio NLL: C=0.8  | M=1  | N_ph=4.8e+06 ... Fatto in 4.0s | Conv: 99.8% | MAE_c: 0.0231 | MAE_s: 0.0300 | (Tot: 0.4 min)
[6/240] Avvio NLL: C=0.8  | M=1  | N_ph=6.0e+06 ... Fatto in 7.9s | Conv: 99.8% | MAE_c: 0.0218 | MAE_s: 0.0286 | (Tot: 0.5 min)
[7/240] Avvio NLL: C=0.5  | M=1  | N_ph=3.0e+05 ... Fatto in 4.8s | Conv: 99.6% | MAE_c: 0.0326 | MAE_s: 0.0425 | (Tot: 0.6 min)
[8/240] Avvio NLL: C=0.5  | M=1  | N_ph=6.0e+05 ... Fatto in 4.0s | Conv: 99.6% | MAE_c: 0.0287 | MAE_s: 0.0384 | (Tot: 0.7 min)
[9/240] Avvio NLL: C=0.5  | M=1  | N_ph=1.2e+06 ... Fatto in 4.1s | Conv: 99.7% | MAE_c: 0.0261 | MAE_s: 0.0353 | (Tot: 0.7 min)
[10/240] Avvio NLL: C=0.5  | M=1  | N_ph=2.4e+06 ... Fatto in 4.6s | Conv: 99.8% | MAE_c: 0.0229 | MAE_s: 0.0314 | (Tot: 0.8 min)
[11/240] Avvio NLL: C=0.5  | M=1  | N_ph=4.8e+06 ... Fatto in 5.1s | Conv: 99.7% | MAE_c: 0.0213 | MAE_s: 0.0296 | (Tot: 0.9 min)
[12/240] Avvio NLL: C=0.5  | M=1  | N_ph=6.0e+06 ... Fatto in 4.5s | Conv: 99.9% | MAE_c: 0.0203 | MAE_s: 0.0278 | (Tot: 1.0 min)
[13/240] Avvio NLL: C=0.2  | M=1  | N_ph=3.0e+05 ... Fatto in 6.6s | Conv: 98.0% | MAE_c: 0.0389 | MAE_s: 0.0518 | (Tot: 1.1 min)
[14/240] Avvio NLL: C=0.2  | M=1  | N_ph=6.0e+05 ... Fatto in 7.0s | Conv: 97.1% | MAE_c: 0.0302 | MAE_s: 0.0411 | (Tot: 1.2 min)
[15/240] Avvio NLL: C=0.2  | M=1  | N_ph=1.2e+06 ... Fatto in 7.0s | Conv: 97.4% | MAE_c: 0.0254 | MAE_s: 0.0346 | (Tot: 1.3 min)
[16/240] Avvio NLL: C=0.2  | M=1  | N_ph=2.4e+06 ... Fatto in 8.6s | Conv: 98.1% | MAE_c: 0.0229 | MAE_s: 0.0317 | (Tot: 1.5 min)
[17/240] Avvio NLL: C=0.2  | M=1  | N_ph=4.8e+06 ... Fatto in 8.4s | Conv: 98.9% | MAE_c: 0.0201 | MAE_s: 0.0283 | (Tot: 1.6 min)
[18/240] Avvio NLL: C=0.2  | M=1  | N_ph=6.0e+06 ... Fatto in 7.6s | Conv: 98.9% | MAE_c: 0.0192 | MAE_s: 0.0275 | (Tot: 1.7 min)
[19/240] SALTO (file esistente): contrast=0.1/mc_trials=1/fit_nll_Nph300000.csv
[20/240] Avvio NLL: C=0.1  | M=1  | N_ph=6.0e+05 ... Fatto in 9.9s | Conv: 92.4% | MAE_c: 0.0384 | MAE_s: 0.0520 | (Tot: 1.9 min)
[21/240] Avvio NLL: C=0.1  | M=1  | N_ph=1.2e+06 ... Fatto in 10.0s | Conv: 91.8% | MAE_c: 0.0276 | MAE_s: 0.0382 | (Tot: 2.1 min)
[22/240] SALTO (file esistente): contrast=0.1/mc_trials=1/fit_nll_Nph2400000.csv
[23/240] Avvio NLL: C=0.1  | M=1  | N_ph=4.8e+06 ... Fatto in 8.7s | Conv: 94.8% | MAE_c: 0.0180 | MAE_s: 0.0260 | (Tot: 2.2 min)
[24/240] Avvio NLL: C=0.1  | M=1  | N_ph=6.0e+06 ... Fatto in 8.6s | Conv: 95.2% | MAE_c: 0.0182 | MAE_s: 0.0267 | (Tot: 2.4 min)
[25/240] Avvio NLL: C=0.08 | M=1  | N_ph=3.0e+05 ... Fatto in 10.5s | Conv: 91.0% | MAE_c: 0.0587 | MAE_s: 0.0782 | (Tot: 2.6 min)
[26/240] Avvio NLL: C=0.08 | M=1  | N_ph=6.0e+05 ... Fatto in 10.0s | Conv: 88.8% | MAE_c: 0.0411 | MAE_s: 0.0558 | (Tot: 2.7 min)
[27/240] Avvio NLL: C=0.08 | M=1  | N_ph=1.2e+06 ... Fatto in 12.6s | Conv: 88.5% | MAE_c: 0.0313 | MAE_s: 0.0441 | (Tot: 2.9 min)
[28/240] Avvio NLL: C=0.08 | M=1  | N_ph=2.4e+06 ... Fatto in 11.2s | Conv: 90.2% | MAE_c: 0.0234 | MAE_s: 0.0332 | (Tot: 3.1 min)
[29/240] Avvio NLL: C=0.08 | M=1  | N_ph=4.8e+06 ... Fatto in 11.8s | Conv: 91.9% | MAE_c: 0.0186 | MAE_s: 0.0275 | (Tot: 3.3 min)
[30/240] Avvio NLL: C=0.08 | M=1  | N_ph=6.0e+06 ... Fatto in 11.8s | Conv: 92.8% | MAE_c: 0.0193 | MAE_s: 0.0287 | (Tot: 3.5 min)
[31/240] SALTO (file esistente): contrast=0.05/mc_trials=1/fit_nll_Nph300000.csv
[32/240] Avvio NLL: C=0.05 | M=1  | N_ph=6.0e+05 ... Fatto in 13.0s | Conv: 77.3% | MAE_c: 0.0608 | MAE_s: 0.0855 | (Tot: 3.7 min)
[33/240] Avvio NLL: C=0.05 | M=1  | N_ph=1.2e+06 ... Fatto in 13.2s | Conv: 76.0% | MAE_c: 0.0422 | MAE_s: 0.0595 | (Tot: 4.0 min)
[34/240] SALTO (file esistente): contrast=0.05/mc_trials=1/fit_nll_Nph2400000.csv
[35/240] Avvio NLL: C=0.05 | M=1  | N_ph=4.8e+06 ... Fatto in 23.6s | Conv: 78.9% | MAE_c: 0.0220 | MAE_s: 0.0330 | (Tot: 4.4 min)
[36/240] Avvio NLL: C=0.05 | M=1  | N_ph=6.0e+06 ... Fatto in 17.2s | Conv: 80.0% | MAE_c: 0.0223 | MAE_s: 0.0338 | (Tot: 4.6 min)
[37/240] Avvio NLL: C=0.02 | M=1  | N_ph=3.0e+05 ... Fatto in 17.2s | Conv: 76.9% | MAE_c: 0.1029 | MAE_s: 0.1711 | (Tot: 4.9 min)
[38/240] Avvio NLL: C=0.02 | M=1  | N_ph=6.0e+05 ... Fatto in 15.3s | Conv: 59.2% | MAE_c: 0.0992 | MAE_s: 0.1613 | (Tot: 5.2 min)
[39/240] Avvio NLL: C=0.02 | M=1  | N_ph=1.2e+06 ... Fatto in 15.1s | Conv: 43.6% | MAE_c: 0.0890 | MAE_s: 0.1473 | (Tot: 5.4 min)
[40/240] Avvio NLL: C=0.02 | M=1  | N_ph=2.4e+06 ... Fatto in 16.3s | Conv: 37.6% | MAE_c: 0.0711 | MAE_s: 0.1274 | (Tot: 5.7 min)
[41/240] Avvio NLL: C=0.02 | M=1  | N_ph=4.8e+06 ... Fatto in 15.5s | Conv: 35.4% | MAE_c: 0.0557 | MAE_s: 0.1063 | (Tot: 6.0 min)
[42/240] Avvio NLL: C=0.02 | M=1  | N_ph=6.0e+06 ... Fatto in 15.6s | Conv: 35.2% | MAE_c: 0.0489 | MAE_s: 0.0970 | (Tot: 6.2 min)
[43/240] Avvio NLL: C=0.01 | M=1  | N_ph=3.0e+05 ... Fatto in 15.8s | Conv: 72.7% | MAE_c: 0.1065 | MAE_s: 0.2113 | (Tot: 6.5 min)
[44/240] Avvio NLL: C=0.01 | M=1  | N_ph=6.0e+05 ... Fatto in 15.8s | Conv: 51.9% | MAE_c: 0.1110 | MAE_s: 0.2047 | (Tot: 6.8 min)
[45/240] Avvio NLL: C=0.01 | M=1  | N_ph=1.2e+06 ... Fatto in 15.5s | Conv: 34.1% | MAE_c: 0.1062 | MAE_s: 0.2102 | (Tot: 7.0 min)
[46/240] Avvio NLL: C=0.01 | M=1  | N_ph=2.4e+06 ... Fatto in 15.4s | Conv: 25.4% | MAE_c: 0.0994 | MAE_s: 0.2301 | (Tot: 7.3 min)
[47/240] Avvio NLL: C=0.01 | M=1  | N_ph=4.8e+06 ... Fatto in 15.1s | Conv: 20.6% | MAE_c: 0.0843 | MAE_s: 0.2267 | (Tot: 7.5 min)
[48/240] Avvio NLL: C=0.01 | M=1  | N_ph=6.0e+06 ... Fatto in 15.5s | Conv: 20.9% | MAE_c: 0.0789 | MAE_s: 0.2359 | (Tot: 7.8 min)
[49/240] Avvio NLL: C=0.8  | M=5  | N_ph=3.0e+05 ... Fatto in 32.2s | Conv: 100.0% | MAE_c: 0.0025 | MAE_s: 0.0039 | (Tot: 8.3 min)
[50/240] Avvio NLL: C=0.8  | M=5  | N_ph=6.0e+05 ... Fatto in 32.4s | Conv: 100.0% | MAE_c: 0.0022 | MAE_s: 0.0034 | (Tot: 8.9 min)
[51/240] Avvio NLL: C=0.8  | M=5  | N_ph=1.2e+06 ... Fatto in 32.5s | Conv: 100.0% | MAE_c: 0.0020 | MAE_s: 0.0031 | (Tot: 9.4 min)
[52/240] Avvio NLL: C=0.8  | M=5  | N_ph=2.4e+06 ... Fatto in 33.3s | Conv: 100.0% | MAE_c: 0.0019 | MAE_s: 0.0030 | (Tot: 10.0 min)
[53/240] Avvio NLL: C=0.8  | M=5  | N_ph=4.8e+06 ... Fatto in 33.6s | Conv: 100.0% | MAE_c: 0.0018 | MAE_s: 0.0027 | (Tot: 10.5 min)
[54/240] Avvio NLL: C=0.8  | M=5  | N_ph=6.0e+06 ... Fatto in 34.1s | Conv: 100.0% | MAE_c: 0.0014 | MAE_s: 0.0021 | (Tot: 11.1 min)
[55/240] Avvio NLL: C=0.5  | M=5  | N_ph=3.0e+05 ... Fatto in 30.2s | Conv: 100.0% | MAE_c: 0.0030 | MAE_s: 0.0046 | (Tot: 11.6 min)
[56/240] Avvio NLL: C=0.5  | M=5  | N_ph=6.0e+05 ... Fatto in 30.7s | Conv: 100.0% | MAE_c: 0.0018 | MAE_s: 0.0028 | (Tot: 12.1 min)
[57/240] Avvio NLL: C=0.5  | M=5  | N_ph=1.2e+06 ... Fatto in 29.9s | Conv: 100.0% | MAE_c: 0.0016 | MAE_s: 0.0025 | (Tot: 12.6 min)
[58/240] Avvio NLL: C=0.5  | M=5  | N_ph=2.4e+06 ... Fatto in 30.9s | Conv: 100.0% | MAE_c: 0.0013 | MAE_s: 0.0020 | (Tot: 13.2 min)
[59/240] Avvio NLL: C=0.5  | M=5  | N_ph=4.8e+06 ... Fatto in 32.2s | Conv: 100.0% | MAE_c: 0.0007 | MAE_s: 0.0011 | (Tot: 13.7 min)
[60/240] Avvio NLL: C=0.5  | M=5  | N_ph=6.0e+06 ... Fatto in 33.0s | Conv: 100.0% | MAE_c: 0.0010 | MAE_s: 0.0016 | (Tot: 14.2 min)
[61/240] Avvio NLL: C=0.2  | M=5  | N_ph=3.0e+05 ... Fatto in 37.2s | Conv: 100.0% | MAE_c: 0.0052 | MAE_s: 0.0075 | (Tot: 14.9 min)
[62/240] Avvio NLL: C=0.2  | M=5  | N_ph=6.0e+05 ... Fatto in 38.8s | Conv: 100.0% | MAE_c: 0.0034 | MAE_s: 0.0051 | (Tot: 15.5 min)
[63/240] Avvio NLL: C=0.2  | M=5  | N_ph=1.2e+06 ... Fatto in 40.8s | Conv: 100.0% | MAE_c: 0.0019 | MAE_s: 0.0029 | (Tot: 16.2 min)
[64/240] Avvio NLL: C=0.2  | M=5  | N_ph=2.4e+06 ... Fatto in 40.5s | Conv: 100.0% | MAE_c: 0.0016 | MAE_s: 0.0024 | (Tot: 16.9 min)
[65/240] Avvio NLL: C=0.2  | M=5  | N_ph=4.8e+06 ... Fatto in 41.4s | Conv: 100.0% | MAE_c: 0.0012 | MAE_s: 0.0018 | (Tot: 17.6 min)
[66/240] Avvio NLL: C=0.2  | M=5  | N_ph=6.0e+06 ... Fatto in 42.4s | Conv: 100.0% | MAE_c: 0.0010 | MAE_s: 0.0014 | (Tot: 18.3 min)
[67/240] SALTO (file esistente): contrast=0.1/mc_trials=5/fit_nll_Nph300000.csv
[68/240] Avvio NLL: C=0.1  | M=5  | N_ph=6.0e+05 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 51.7s | Conv: 100.0% | MAE_c: 0.0068 | MAE_s: 0.0098 | (Tot: 19.1 min)
[69/240] Avvio NLL: C=0.1  | M=5  | N_ph=1.2e+06 ... Fatto in 51.9s | Conv: 99.9% | MAE_c: 0.0041 | MAE_s: 0.0059 | (Tot: 20.0 min)
[70/240] SALTO (file esistente): contrast=0.1/mc_trials=5/fit_nll_Nph2400000.csv
[71/240] Avvio NLL: C=0.1  | M=5  | N_ph=4.8e+06 ... Fatto in 55.0s | Conv: 100.0% | MAE_c: 0.0016 | MAE_s: 0.0024 | (Tot: 20.9 min)
[72/240] Avvio NLL: C=0.1  | M=5  | N_ph=6.0e+06 ... Fatto in 56.1s | Conv: 100.0% | MAE_c: 0.0016 | MAE_s: 0.0024 | (Tot: 21.9 min)
[73/240] Avvio NLL: C=0.08 | M=5  | N_ph=3.0e+05 ... Fatto in 56.8s | Conv: 99.9% | MAE_c: 0.0189 | MAE_s: 0.0265 | (Tot: 22.8 min)
[74/240] Avvio NLL: C=0.08 | M=5  | N_ph=6.0e+05 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 58.8s | Conv: 99.9% | MAE_c: 0.0101 | MAE_s: 0.0143 | (Tot: 23.8 min)
[75/240] Avvio NLL: C=0.08 | M=5  | N_ph=1.2e+06 ... Fatto in 60.6s | Conv: 99.9% | MAE_c: 0.0058 | MAE_s: 0.0084 | (Tot: 24.8 min)
[76/240] Avvio NLL: C=0.08 | M=5  | N_ph=2.4e+06 ... E VariableMetricBuilder Initial matrix not pos.def.
E VariableMetricBuilder Initial matrix not pos.def.
E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 63.1s | Conv: 99.9% | MAE_c: 0.0033 | MAE_s: 0.0048 | (Tot: 25.9 min)
[77/240] Avvio NLL: C=0.08 | M=5  | N_ph=4.8e+06 ... Fatto in 65.0s | Conv: 100.0% | MAE_c: 0.0023 | MAE_s: 0.0033 | (Tot: 27.0 min)
[78/240] Avvio NLL: C=0.08 | M=5  | N_ph=6.0e+06 ... Fatto in 65.6s | Conv: 100.0% | MAE_c: 0.0023 | MAE_s: 0.0035 | (Tot: 28.0 min)
[79/240] SALTO (file esistente): contrast=0.05/mc_trials=5/fit_nll_Nph300000.csv
[80/240] Avvio NLL: C=0.05 | M=5  | N_ph=6.0e+05 ... Fatto in 72.4s | Conv: 99.3% | MAE_c: 0.0312 | MAE_s: 0.0443 | (Tot: 29.3 min)
[81/240] Avvio NLL: C=0.05 | M=5  | N_ph=1.2e+06 ... Fatto in 73.5s | Conv: 99.1% | MAE_c: 0.0166 | MAE_s: 0.0235 | (Tot: 30.5 min)
[82/240] SALTO (file esistente): contrast=0.05/mc_trials=5/fit_nll_Nph2400000.csv
[83/240] Avvio NLL: C=0.05 | M=5  | N_ph=4.8e+06 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 79.0s | Conv: 99.2% | MAE_c: 0.0063 | MAE_s: 0.0092 | (Tot: 31.8 min)
[84/240] Avvio NLL: C=0.05 | M=5  | N_ph=6.0e+06 ... Fatto in 81.9s | Conv: 99.4% | MAE_c: 0.0056 | MAE_s: 0.0083 | (Tot: 33.2 min)
[85/240] Avvio NLL: C=0.02 | M=5  | N_ph=3.0e+05 ... Fatto in 75.9s | Conv: 99.5% | MAE_c: 0.1100 | MAE_s: 0.1628 | (Tot: 34.4 min)
[86/240] Avvio NLL: C=0.02 | M=5  | N_ph=6.0e+05 ... Fatto in 80.9s | Conv: 96.6% | MAE_c: 0.0997 | MAE_s: 0.1454 | (Tot: 35.8 min)
[87/240] Avvio NLL: C=0.02 | M=5  | N_ph=1.2e+06 ... Fatto in 81.5s | Conv: 89.6% | MAE_c: 0.0848 | MAE_s: 0.1257 | (Tot: 37.1 min)
[88/240] Avvio NLL: C=0.02 | M=5  | N_ph=2.4e+06 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 80.8s | Conv: 83.7% | MAE_c: 0.0615 | MAE_s: 0.0964 | (Tot: 38.5 min)
[89/240] Avvio NLL: C=0.02 | M=5  | N_ph=4.8e+06 ... Fatto in 82.0s | Conv: 81.3% | MAE_c: 0.0451 | MAE_s: 0.0733 | (Tot: 39.9 min)
[90/240] Avvio NLL: C=0.02 | M=5  | N_ph=6.0e+06 ... Fatto in 84.1s | Conv: 80.6% | MAE_c: 0.0402 | MAE_s: 0.0674 | (Tot: 41.3 min)
[91/240] Avvio NLL: C=0.01 | M=5  | N_ph=3.0e+05 ... Fatto in 78.1s | Conv: 99.0% | MAE_c: 0.1263 | MAE_s: 0.2140 | (Tot: 42.6 min)
[92/240] Avvio NLL: C=0.01 | M=5  | N_ph=6.0e+05 ... E VariableMetricBuilder Initial matrix not pos.def.
[93/240] Avvio NLL: C=0.01 | M=5  | N_ph=1.2e+06 ... Fatto in 83.7s | Conv: 81.7% | MAE_c: 0.1153 | MAE_s: 0.1845 | (Tot: 1.4 min)
[94/240] Avvio NLL: C=0.01 | M=5  | N_ph=2.4e+06 ... E VariableMetricBuilder Initial matrix not pos.def.
E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 75.8s | Conv: 69.8% | MAE_c: 0.1047 | MAE_s: 0.1765 | (Tot: 2.7 min)
[95/240] Avvio NLL: C=0.01 | M=5  | N_ph=4.8e+06 ... Fatto in 69.0s | Conv: 61.6% | MAE_c: 0.0901 | MAE_s: 0.1595 | (Tot: 3.8 min)
[96/240] Avvio NLL: C=0.01 | M=5  | N_ph=6.0e+06 ... Fatto in 73.6s | Conv: 61.2% | MAE_c: 0.0834 | MAE_s: 0.1551 | (Tot: 5.0 min)
[97/240] Avvio NLL: C=0.8  | M=10 | N_ph=3.0e+05 ... Fatto in 65.2s | Conv: 100.0% | MAE_c: 0.0003 | MAE_s: 0.0005 | (Tot: 6.1 min)
[98/240] Avvio NLL: C=0.8  | M=10 | N_ph=6.0e+05 ... Fatto in 66.9s | Conv: 100.0% | MAE_c: 0.0002 | MAE_s: 0.0003 | (Tot: 7.3 min)
[99/240] Avvio NLL: C=0.8  | M=10 | N_ph=1.2e+06 ... Fatto in 66.6s | Conv: 100.0% | MAE_c: 0.0001 | MAE_s: 0.0002 | (Tot: 8.4 min)
[100/240] Avvio NLL: C=0.8  | M=10 | N_ph=2.4e+06 ... Fatto in 66.7s | Conv: 100.0% | MAE_c: 0.0001 | MAE_s: 0.0002 | (Tot: 9.5 min)
[101/240] Avvio NLL: C=0.8  | M=10 | N_ph=4.8e+06 ... Fatto in 70.8s | Conv: 100.0% | MAE_c: 0.0001 | MAE_s: 0.0001 | (Tot: 10.7 min)
[102/240] Avvio NLL: C=0.8  | M=10 | N_ph=6.0e+06 ... Fatto in 68.7s | Conv: 100.0% | MAE_c: 0.0001 | MAE_s: 0.0001 | (Tot: 11.8 min)
[103/240] Avvio NLL: C=0.5  | M=10 | N_ph=3.0e+05 ... Fatto in 60.5s | Conv: 100.0% | MAE_c: 0.0005 | MAE_s: 0.0007 | (Tot: 12.8 min)
[104/240] Avvio NLL: C=0.5  | M=10 | N_ph=6.0e+05 ... Fatto in 59.3s | Conv: 100.0% | MAE_c: 0.0004 | MAE_s: 0.0006 | (Tot: 13.8 min)
[105/240] Avvio NLL: C=0.5  | M=10 | N_ph=1.2e+06 ... Fatto in 59.6s | Conv: 100.0% | MAE_c: 0.0002 | MAE_s: 0.0003 | (Tot: 14.8 min)
[106/240] Avvio NLL: C=0.5  | M=10 | N_ph=2.4e+06 ... Fatto in 60.6s | Conv: 100.0% | MAE_c: 0.0002 | MAE_s: 0.0003 | (Tot: 15.8 min)
[107/240] Avvio NLL: C=0.5  | M=10 | N_ph=4.8e+06 ... Fatto in 64.4s | Conv: 100.0% | MAE_c: 0.0002 | MAE_s: 0.0003 | (Tot: 16.9 min)
[108/240] Avvio NLL: C=0.5  | M=10 | N_ph=6.0e+06 ... Fatto in 65.6s | Conv: 100.0% | MAE_c: 0.0001 | MAE_s: 0.0002 | (Tot: 18.0 min)
[109/240] Avvio NLL: C=0.2  | M=10 | N_ph=3.0e+05 ... Fatto in 72.6s | Conv: 100.0% | MAE_c: 0.0017 | MAE_s: 0.0023 | (Tot: 19.2 min)
[110/240] Avvio NLL: C=0.2  | M=10 | N_ph=6.0e+05 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 74.3s | Conv: 100.0% | MAE_c: 0.0011 | MAE_s: 0.0015 | (Tot: 20.4 min)
[111/240] Avvio NLL: C=0.2  | M=10 | N_ph=1.2e+06 ... E VariableMetricBuilder Initial matrix not pos.def.
E VariableMetricBuilder Initial matrix not pos.def.
E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 77.9s | Conv: 100.0% | MAE_c: 0.0007 | MAE_s: 0.0010 | (Tot: 21.7 min)
[112/240] Avvio NLL: C=0.2  | M=10 | N_ph=2.4e+06 ... Fatto in 78.9s | Conv: 100.0% | MAE_c: 0.0005 | MAE_s: 0.0007 | (Tot: 23.1 min)
[113/240] Avvio NLL: C=0.2  | M=10 | N_ph=4.8e+06 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 82.0s | Conv: 100.0% | MAE_c: 0.0004 | MAE_s: 0.0005 | (Tot: 24.4 min)
[114/240] Avvio NLL: C=0.2  | M=10 | N_ph=6.0e+06 ... Fatto in 81.9s | Conv: 100.0% | MAE_c: 0.0003 | MAE_s: 0.0005 | (Tot: 25.8 min)
[115/240] SALTO (file esistente): contrast=0.1/mc_trials=10/fit_nll_Nph300000.csv
[116/240] Avvio NLL: C=0.1  | M=10 | N_ph=6.0e+05 ... Fatto in 101.3s | Conv: 100.0% | MAE_c: 0.0027 | MAE_s: 0.0037 | (Tot: 27.5 min)
[117/240] Avvio NLL: C=0.1  | M=10 | N_ph=1.2e+06 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 101.5s | Conv: 100.0% | MAE_c: 0.0017 | MAE_s: 0.0023 | (Tot: 29.2 min)
[118/240] SALTO (file esistente): contrast=0.1/mc_trials=10/fit_nll_Nph2400000.csv
[119/240] Avvio NLL: C=0.1  | M=10 | N_ph=4.8e+06 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 111.6s | Conv: 100.0% | MAE_c: 0.0008 | MAE_s: 0.0011 | (Tot: 31.0 min)
[120/240] Avvio NLL: C=0.1  | M=10 | N_ph=6.0e+06 ... Fatto in 111.4s | Conv: 100.0% | MAE_c: 0.0007 | MAE_s: 0.0010 | (Tot: 32.9 min)
[121/240] Avvio NLL: C=0.08 | M=10 | N_ph=3.0e+05 ... Fatto in 107.5s | Conv: 100.0% | MAE_c: 0.0113 | MAE_s: 0.0158 | (Tot: 34.7 min)
[122/240] Avvio NLL: C=0.08 | M=10 | N_ph=6.0e+05 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 124.9s | Conv: 100.0% | MAE_c: 0.0042 | MAE_s: 0.0058 | (Tot: 36.8 min)
[123/240] Avvio NLL: C=0.08 | M=10 | N_ph=1.2e+06 ... Fatto in 126.1s | Conv: 100.0% | MAE_c: 0.0023 | MAE_s: 0.0031 | (Tot: 38.9 min)
[124/240] Avvio NLL: C=0.08 | M=10 | N_ph=2.4e+06 ... Fatto in 130.1s | Conv: 100.0% | MAE_c: 0.0015 | MAE_s: 0.0021 | (Tot: 41.1 min)
[125/240] Avvio NLL: C=0.08 | M=10 | N_ph=4.8e+06 ... Fatto in 136.5s | Conv: 100.0% | MAE_c: 0.0010 | MAE_s: 0.0014 | (Tot: 43.3 min)
[126/240] Avvio NLL: C=0.08 | M=10 | N_ph=6.0e+06 ... Fatto in 138.5s | Conv: 100.0% | MAE_c: 0.0009 | MAE_s: 0.0012 | (Tot: 45.6 min)
[127/240] SALTO (file esistente): contrast=0.05/mc_trials=10/fit_nll_Nph300000.csv
[128/240] Avvio NLL: C=0.05 | M=10 | N_ph=6.0e+05 ... E VariableMetricBuilder Initial matrix not pos.def.
Fatto in 143.4s | Conv: 99.9% | MAE_c: 0.0199 | MAE_s: 0.0280 | (Tot: 48.0 min)
[129/240] Avvio NLL: C=0.05 | M=10 | N_ph=1.2e+06 ... Fatto in 148.9s | Conv: 99.9% | MAE_c: 0.0082 | MAE_s: 0.0116 | (Tot: 50.5 min)
[130/240] SALTO (file esistente): contrast=0.05/mc_trials=10/fit_nll_Nph2400000.csv
[131/240] Avvio NLL: C=0.05 | M=10 | N_ph=4.8e+06 ... Fatto in 162.8s | Conv: 100.0% | MAE_c: 0.0022 | MAE_s: 0.0031 | (Tot: 53.2 min)
[132/240] Avvio NLL: C=0.05 | M=10 | N_ph=6.0e+06 ...
````